In [ ]:
# Importing libraries

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import shap
from scipy.stats import spearmanr

In [ ]:
# setting colours

plt.rcParams['font.family'] = 'DejaVu Sans'
GREEN, ORANGE, RED = '#508C78', '#e0a530', '#BE4135'

In [ ]:
# Matching colours to labels

SETS = [('Tier 1', GREEN),
        ('Tier 1 + 2', ORANGE),
        ('All tiers', RED,)]

In [ ]:
# Loading in text set answers
y_test = np.load('y_test.npy')

# Turning into an array
y_test = np.asarray(y_test)

In [ ]:
# Loading in results of the tiers

scores = [np.load('score_tier1.npy'), np.load('score_tier12.npy'), np.load('score_all.npy')]

In [ ]:
# Generation of the ROC curve

# creating figure
fig, axs = plt.subplots(figsize=(5.5, 5.5))

for (label, colour), sc in zip(SETS, scores):
    fpr, tpr, _ = roc_curve(y_test, sc)
    axs.plot(fpr, tpr, color=colour, lw=2.0, label=f'{label.split(" (")[0]} (AUC = {auc(fpr,tpr):.2f})')

axs.plot([0, 1], [0, 1], ls=(0, (5, 4)), lw=1.3, color='black', label='Chance')
axs.set_xlim(-0.01, 1.01); axs.set_ylim(-0.01, 1.01); axs.set_box_aspect(1)
axs.set_xlabel('False positive rate', fontsize=10)
axs.set_ylabel('True positive rate', fontsize=10)
axs.set_title('Receiver operating characteristic', fontsize=11.5,
              fontweight='bold', color='black', pad=8)
axs.legend(frameon=False, fontsize=8.3, loc='lower right')
plt.tight_layout()

In [ ]:
# Generation of the PR curve

# creating figure
fig, axs = plt.subplots(figsize=(5.5, 5.5))

for (label, colour), sc in zip(SETS, scores):
    prec, rec, _ = precision_recall_curve(y_test, sc)
    axs.plot(rec, prec, color=colour, lw=2.0, label=f'{label.split(" (")[0]} (AP = {average_precision_score(y_test,sc):.2f})')

axs.axhline(prev, ls=(0,(5,4)), lw=1.3, color= 'black', label=f'Baseline ({prev:.2f})')
axs.set_xlim(-0.01,1.01); axp.set_ylim(prev-0.05,1.02); axp.set_box_aspect(1)
axs.set_xlabel('Recall', fontsize=10)
axs.set_ylabel('Precision', fontsize=10)
axs.set_title('Precision\u2013recall', fontsize=11.5, fontweight='bold', color= 'black', pad=8)
axs.legend(frameon=False, fontsize=8.3, loc='upper right')
plt.tight_layout()

In [ ]:
# Loading in SHAP features

shap_values = np.load('shap_all.npy')                 
feature_names = np.load('shap_features.npy', allow_pickle= True) 

In [ ]:
# Labelling proteins as their tiers

TIER = {'PILRA': 1, 'PILRB': 1, 'CD2AP': 1,
        'APOE': 2, 'GRN': 2, 'PRSS8': 2, 'BLNK': 2}
TIER_COLOURS = {1: GREEN, 2: ORANGE, 3: RED}
DEMOGRAPHIC = {'Age', 'Age²', 'Sex', 'Age-by-sex interaction'}

In [ ]:
# Quick function for the demographics 

def colour_for(f):
    return 'grey' if f in DEMOGRAPHIC else TIER_COLOURS[TIER.get(f, 3)]

In [ ]:
# Writing function that converts to short name
def short_name(label):
    if label.endswith(')') and '(' in label:
        return label[label.rfind('(') + 1 : -1]
    return label

In [ ]:
# Applying function
feature_names = list(map(short_name, feature_names))

In [ ]:
# finding mean for all features

mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names)

In [ ]:
# Plot


plot_df = mean_shap.sort_values()
fig, ax = plt.subplots(figsize=(5.5, 7))
ax.barh(plot_df.index, plot_df.values,
        color=[colour_for(f) for f in plot_df.index], height=0.7)

age_val = mean_shap['Age']
ax.axvline(age_val, ls=(0, (5, 4)), lw=1.3, color='black')

ax.set_xlabel('Mean SHAP value', fontsize=10)
ax.set_title('Protein contributions to classification', fontsize=11.5,
             fontweight='bold', color='black', pad=8)
ax.tick_params(axis='y', labelsize=8.5)
ax.spines[['top', 'right']].set_visible(False)

handles = [Patch(color=GREEN,  label='Tier 1'),
           Patch(color=ORANGE, label='Tier 2'),
           Patch(color=RED,    label='Tier 3'),
           Patch(color='grey',   label='Demographic'),
           Line2D([0], [0], ls=(0, (5, 4)), color='black', lw=1.3, label='Age')]
ax.legend(handles=handles, frameon=False, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('shap_bar.pdf', bbox_inches='tight')
plt.show()

In [ ]:
print(list(X_test.columns) == list(X_test_short.columns.map(lambda c: c)))  # same length/order
for raw, new in zip(X_test.columns, X_test_short.columns):
    if 'age' in raw.lower() or 'sex' in raw.lower():
        print(raw, '->', new)

In [ ]:
# Plot: predictive contribution by tier

TIER = {'PILRA': 1, 'PILRB': 1, 'CD2AP': 1,
        'APOE': 2, 'GRN': 2, 'PRSS8': 2, 'BLNK': 2}
TIER_COLOURS = {1: GREEN, 2: ORANGE, 3: RED}
demographic = ['Age', 'Age²', 'Sex', 'Age-by-sex interaction']

proteins = mean_shap.drop(demographic)

fig, ax = plt.subplots(figsize=(5.5, 7))

for p, val in proteins.items():
    t = TIER.get(p, 3)
    ax.scatter(t, val, color=TIER_COLOURS[t], alpha=0.6, s=50, zorder=3)
    if p in ['APP', 'SERPINF2']:
        ax.annotate(p, (t, val), xytext=(6, 0), textcoords='offset points',
                    fontsize=7.5, va='center')

age_val = mean_shap['Age']
ax.axhline(age_val, ls=(0, (5, 4)), lw=1.3, color='black')

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Tier 1', 'Tier 2', 'Tier 3'])
ax.set_xlim(0.5, 3.8)
ax.set_xlabel('Causal confidence (highest → lowest)', fontsize=10)
ax.set_ylabel('Mean SHAP value', fontsize=10)
ax.set_title('Causal confidence vs predictive contribution', fontsize=11.5,
             fontweight='bold', color='black', pad=8)
ax.spines[['top', 'right']].set_visible(False)

handles = [Line2D([0], [0], ls=(0, (5, 4)), color='black', lw=1.3, label='Age')]
ax.legend(handles=handles, frameon=False, fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('tier_vs_shap.pdf', bbox_inches='tight')
plt.show()